# LAB 09 — Tokenizer và label alignment

Tokenizer: `bert-base-cased` (WordPiece, giữ chữ hoa), cùng tokenizer dùng trong `ner_finetuning.ipynb`.

1. Từ word-level label đến token-level label: `tokens`, `input_ids`, `attention_mask`, `word_ids`, `labels`;
2. Strategy A (subword đầu + IGNORE) so với Strategy B (lan truyền I-X);
3. IGNORE trong loss;
4. **Exp 1** — thống kê tokenization trên CoNLL-2003 test: R = #subword tokens / #words, câu ngắn / dài, entity / non-entity, loại từ nào bị tách.

In [1]:
import os, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"; os.environ["TRANSFORMERS_VERBOSITY"] = "error"
warnings.filterwarnings("ignore")
from collections import Counter
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer
from baseline_ner import LABELS, IGNORE, load_conll, align_labels, bio_to_spans
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 120)
tok = AutoTokenizer.from_pretrained("bert-base-cased")
print(type(tok).__name__, "| vocab", tok.vocab_size, "| labels:", LABELS)

BertTokenizer | vocab 28996 | labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


## 1. Từ word-level label đến token-level label

In [2]:
def show(words, tags, strategy="A"):
    enc = tok(words, is_split_into_words=True, truncation=True, max_length=128)
    wids = enc.word_ids()
    lab = align_labels(wids, tags, strategy)
    return pd.DataFrame({"token": tok.convert_ids_to_tokens(enc["input_ids"]), "input_id": enc["input_ids"],
                         "attention_mask": enc["attention_mask"], "word_id": wids,
                         "word": [words[w] if w is not None else "" for w in wids],
                         "word label": [tags[w] if w is not None else "" for w in wids],
                         f"label {strategy}": [LABELS[l] if l != IGNORE else "IGNORE (-100)" for l in lab]})

show("Barack Obama visited Paris .".split(), ["B-PER", "I-PER", "O", "B-LOC", "O"])

,token,input_id,attention_mask,word_id,word,word label,label A
0,[CLS],101,1,NaN,,,IGNORE (-100)
1,Barack,14319,1,0.0,Barack,B-PER,B-PER
2,Obama,7661,1,1.0,Obama,I-PER,I-PER
3,visited,3891,1,2.0,visited,O,O
4,Paris,2123,1,3.0,Paris,B-LOC,B-LOC
5,.,119,1,4.0,.,O,O
6,[SEP],102,1,NaN,,,IGNORE (-100)


Câu này không có từ nào bị tách: mỗi word ứng với đúng 1 token. Chỉ có `[CLS]` và `[SEP]` là token không thuộc word nào (`word_id = None`), nên nhận IGNORE.

Calculation Exercise 1 (Elon Musk founded SpaceX in California.), kiểm tra bằng tokenizer thật:

In [3]:
w1 = "Elon Musk founded SpaceX in California .".split()
t1 = ["B-PER", "I-PER", "O", "B-ORG", "O", "B-LOC", "O"]
a = show(w1, t1, "A"); a["label B"] = show(w1, t1, "B")["label B"]
a

,token,input_id,attention_mask,word_id,word,word label,label A,label B
0,[CLS],101,1,NaN,,,IGNORE (-100),IGNORE (-100)
1,El,2896,1,0.0,Elon,B-PER,B-PER,B-PER
2,##on,1320,1,0.0,Elon,B-PER,IGNORE (-100),I-PER
3,Mu,19569,1,1.0,Musk,I-PER,I-PER,I-PER
4,##sk,5276,1,1.0,Musk,I-PER,IGNORE (-100),I-PER
5,founded,1771,1,2.0,founded,O,O,O
6,Space,4525,1,3.0,SpaceX,B-ORG,B-ORG,B-ORG
7,##X,3190,1,3.0,SpaceX,B-ORG,IGNORE (-100),I-ORG
8,in,1107,1,4.0,in,O,O,O
9,California,1756,1,5.0,California,B-LOC,B-LOC,B-LOC


Cả 3 entity của câu đều có word bị tách: "Elon" → `El ##on`, "Musk" → `Mu ##sk`, "SpaceX" → `Space ##X`. 7 word thành 10 token (R = 1.43).

- **Strategy A:** chỉ subword đầu (`El`, `Mu`, `Space`) nhận nhãn của word; `##on`, `##sk`, `##X` nhận IGNORE. Loss tính trên 7 vị trí, đúng bằng số word.
- **Strategy B:** `##on` nhận I-PER, `##X` nhận I-ORG. Entity 1 word "SpaceX" thành chuỗi `B-ORG I-ORG` ở mức token, nhưng vẫn là 1 entity. Loss tính trên 10 vị trí.

Mình chọn **Strategy A** làm chính: nhãn gốc là nhãn của word, nên chỉ cần dự đoán một lần cho mỗi word. Cách này không phải tự đặt ra nhãn cho các mảnh `##`, và cách đánh giá khớp với gold (một dự đoán cho mỗi word). Strategy B được train thêm để so sánh trong `ner_finetuning.ipynb`.

In [4]:
w2 = "University of California Berkeley".split()
t2 = ["B-ORG", "I-ORG", "I-ORG", "I-ORG"]
b = show(w2, t2, "A"); b["label B"] = show(w2, t2, "B")["label B"]
print("Calculation Exercise 2 — spans:", bio_to_spans(t2), "| nếu gán ORG cho mọi word thì không còn biết biên")
b

Calculation Exercise 2 — spans: [(0, 4, 'ORG')] | nếu gán ORG cho mọi word thì không còn biết biên


,token,input_id,attention_mask,word_id,word,word label,label A,label B
0,[CLS],101,1,NaN,,,IGNORE (-100),IGNORE (-100)
1,University,1239,1,0.0,University,B-ORG,B-ORG,B-ORG
2,of,1104,1,1.0,of,I-ORG,I-ORG,I-ORG
3,California,1756,1,2.0,California,I-ORG,I-ORG,I-ORG
4,Berkeley,7823,1,3.0,Berkeley,I-ORG,I-ORG,I-ORG
5,[SEP],102,1,NaN,,,IGNORE (-100),IGNORE (-100)


Ví dụ "playing" của đề với một câu CoNLL có từ bị tách nhiều:

In [5]:
data = load_conll()
# câu ngắn đầu tiên trong test có một word thuộc entity bị tách thành ≥ 3 subword
i = next(k for k, (ws, ts) in enumerate(zip(data["test"]["tokens"], data["test"]["tags"]))
         if len(ws) <= 12 and any(t != "O" and len(tok.tokenize(w)) >= 3 for w, t in zip(ws, ts)))
w3, t3 = data["test"]["tokens"][i], data["test"]["tags"][i]
print("playing →", tok.tokenize("playing"), "| Washington →", tok.tokenize("Washington"))
c = show(w3, t3, "A"); c["label B"] = show(w3, t3, "B")["label B"]
c

playing → ['playing'] | Washington → ['Washington']


,token,input_id,attention_mask,word_id,word,word label,label A,label B
0,[CLS],101,1,NaN,,,IGNORE (-100),IGNORE (-100)
1,S,156,1,0.0,SOCCER,O,O,O
2,##OC,9244,1,0.0,SOCCER,O,IGNORE (-100),O
3,##CE,10954,1,0.0,SOCCER,O,IGNORE (-100),O
4,##R,2069,1,0.0,SOCCER,O,IGNORE (-100),O
5,-,118,1,1.0,-,O,O,O
6,J,147,1,2.0,JAPAN,B-LOC,B-LOC,B-LOC
7,##AP,12240,1,2.0,JAPAN,B-LOC,IGNORE (-100),I-LOC
8,##AN,14962,1,2.0,JAPAN,B-LOC,IGNORE (-100),I-LOC
9,GE,25075,1,3.0,GET,O,O,O


Với `bert-base-cased`, "playing" và "Washington" đều là 1 token nguyên; ví dụ `play ##ing` trong đề chỉ là minh họa. Chỉ tên riêng hiếm mới bị tách (như trong câu trên).

## 2. Batch: padding, attention mask, labels

In [6]:
batch_words = [w1, "Paris is in France .".split()]
batch_tags = [t1, ["B-LOC", "O", "O", "B-LOC", "O"]]
enc = tok(batch_words, is_split_into_words=True, truncation=True, padding=True, return_tensors="pt")
labels = torch.full_like(enc["input_ids"], IGNORE)
for k in range(len(batch_words)):
    lab = align_labels(enc.word_ids(k), batch_tags[k], "A")
    labels[k, :len(lab)] = torch.tensor(lab)
for key in ["input_ids", "attention_mask"]:
    print(key, tuple(enc[key].shape)); print(enc[key])
print("labels", tuple(labels.shape)); print(labels)
print("word_ids câu 2:", enc.word_ids(1))

input_ids (2, 12)
tensor([[  101,  2896,  1320, 19569,  5276,  1771,  4525,  3190,  1107,  1756,
           119,   102],
        [  101,  2123,  1110,  1107,  1699,   119,   102,     0,     0,     0,
             0,     0]])
attention_mask (2, 12)
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]])
labels (2, 12)
tensor([[-100,    1, -100,    2, -100,    0,    3, -100,    0,    5,    0, -100],
        [-100,    5,    0,    0,    5,    0, -100, -100, -100, -100, -100, -100]])
word_ids câu 2: [None, 0, 1, 2, 3, 4, None, None, None, None, None, None]


`[PAD]` có attention_mask = 0 và `word_id = None`, nên label của nó là IGNORE. Nếu pad label bằng 0 (= `O`), model sẽ bị dạy rằng padding là "ngoài entity", làm lệch loss về phía O.

## 3. IGNORE trong loss

L = −Σ<sub>i: y<sub>i</sub> ≠ IGNORE</sub> log P(y<sub>i</sub> | x), chia trung bình cho số vị trí không bị IGNORE.

In [7]:
torch.manual_seed(0)
logits = torch.randn(labels.shape[0], labels.shape[1], len(LABELS))
ce = torch.nn.functional.cross_entropy(logits.view(-1, len(LABELS)), labels.view(-1), ignore_index=IGNORE)
keep = labels.view(-1) != IGNORE
manual = -torch.log_softmax(logits.view(-1, len(LABELS)), -1)[keep, labels.view(-1)[keep]].mean()
print(f"cross_entropy(ignore_index=-100) = {ce:.6f} | tính tay trên {int(keep.sum())}/{keep.numel()} vị trí = {manual:.6f}")

cross_entropy(ignore_index=-100) = 2.357470 | tính tay trên 12/24 vị trí = 2.357470


Hai cách cho cùng kết quả. Các token IGNORE vẫn đi qua Transformer: chúng có hidden state và các token khác vẫn attend tới chúng. Chỉ là dự đoán ở vị trí đó không được tính vào loss.

## 4. Exp 1 — Tokenization analysis trên CoNLL-2003 test (3 453 câu)

Với mỗi câu: #words, #tokens (bỏ `[CLS]` / `[SEP]`), #subword splits (số word bị tách ≥ 2 mảnh), R = #tokens / #words.

In [8]:
rows, word_rows = [], []
for sid, (ws, ts) in enumerate(zip(data["test"]["tokens"], data["test"]["tags"])):
    pieces = [tok.tokenize(w) for w in ws]
    n_tok = sum(map(len, pieces))
    rows.append({"sid": sid, "words": len(ws), "tokens": n_tok, "splits": sum(len(p) > 1 for p in pieces),
                 "R": n_tok / len(ws), "sentence": " ".join(ws)})
    for w, t, p in zip(ws, ts, pieces):
        word_rows.append({"word": w, "tag": t, "type": t[2:] if t != "O" else "O", "pieces": len(p), "tokens": " ".join(p)})
S, W = pd.DataFrame(rows), pd.DataFrame(word_rows)
print(f"{len(S)} câu, {len(W)} word → {S.tokens.sum()} token | R tổng = {S.tokens.sum() / len(W):.3f} | "
      f"{(W.pieces > 1).mean():.1%} word bị tách")
print(f"câu dài nhất: {S.words.max()} word / {S.tokens.max()} token | câu > 126 token (bị cắt ở T = 128): {(S.tokens > 126).sum()}")

3453 câu, 46435 word → 63461 token | R tổng = 1.367 | 17.8% word bị tách
câu dài nhất: 124 word / 146 token | câu > 126 token (bị cắt ở T = 128): 2


Bảng 20 câu (lấy ngẫu nhiên, seed 42, trong các câu có ≥ 5 word):

In [9]:
sample = S[S.words >= 5].sample(20, random_state=42).sort_values("words")
sample[["words", "tokens", "splits", "R", "sentence"]].round(2)

,words,tokens,splits,R,sentence
1712,6,21,6,3.50,Durum 1168.9 1225.0 44.3 965.3 1069.6
461,6,14,3,2.33,MINNESOTA 7 10 .412 8 1/2
2544,7,17,4,2.43,12. Warwara Zelenskaja ( Russia ) 1:49.66
362,7,11,1,1.57,BUFFALO 13 12 1 77 76 27
2422,7,16,4,2.29,25. Hilary Lindh ( U.S. ) 1:19.41
628,7,7,0,1.00,"70 , Phil Gray 89 ) ."
1187,8,10,2,1.25,Multinational commander going back to east Zaire .
585,8,10,1,1.25,Groningen 18 4 6 8 20 31 18
1865,8,8,0,1.00,German Santa in bank nearly gets arrested .
3245,8,8,0,1.00,"goals for , against , points ) :"


In [10]:
med = S.words.median()
S["độ dài"] = np.where(S.words <= med, f"ngắn (≤ {med:.0f} word)", f"dài (> {med:.0f} word)")
by_len = S.groupby("độ dài").apply(lambda g: pd.Series({"câu": len(g), "word": g.words.sum(), "token": g.tokens.sum(),
                                                         "R": g.tokens.sum() / g.words.sum(),
                                                         "% word bị tách": g.splits.sum() / g.words.sum()}))
W["nhóm"] = np.where(W.type == "O", "non-entity (O)", "entity")
by_ent = W.groupby("nhóm").apply(lambda g: pd.Series({"word": len(g), "R": g.pieces.mean(), "% word bị tách": (g.pieces > 1).mean()}))
by_type = W.groupby("type").apply(lambda g: pd.Series({"word": len(g), "R": g.pieces.mean(), "% word bị tách": (g.pieces > 1).mean()}))
print(by_len.round(3)); print(); print(by_ent.round(3)); print(); print(by_type.round(3))

                    câu     word    token      R  % word bị tách
độ dài                                                          
dài (> 9 word)   1516.0  35531.0  43547.0  1.226           0.127
ngắn (≤ 9 word)  1937.0  10904.0  19914.0  1.826           0.348

                   word      R  % word bị tách
nhóm                                          
entity           8112.0  1.915           0.453
non-entity (O)  38323.0  1.251           0.120

         word      R  % word bị tách
type                                
LOC    1925.0  1.775           0.312
MISC    918.0  1.730           0.312
O     38323.0  1.251           0.120
ORG    2496.0  1.936           0.452
PER    2773.0  2.055           0.599


### Loại từ nào bị tách thành nhiều subword?

In [11]:
def kind(w):
    if any(c.isdigit() for c in w): return "có chữ số"
    if not any(c.isalpha() for c in w): return "dấu câu / ký hiệu"
    if w.isupper() and len(w) > 1: return "VIẾT HOA TOÀN BỘ"
    if w[0].isupper(): return "Viết hoa chữ đầu"
    return "chữ thường"
W["kiểu"] = W.word.map(kind)
W["độ dài từ"] = pd.cut(W.word.str.len(), [0, 4, 8, 12, 100], labels=["1–4", "5–8", "9–12", "13+"])
for col in ["kiểu", "độ dài từ"]:
    print(W.groupby(col, observed=True).apply(lambda g: pd.Series({"word": len(g), "R": g.pieces.mean(),
          "% bị tách": (g.pieces > 1).mean(), "% là entity": (g.type != "O").mean()})).round(3), "\n")
print("Word bị tách nhiều mảnh nhất:")
top = W.drop_duplicates("word").sort_values("pieces", ascending=False).head(15)
print(top[["word", "tag", "pieces", "tokens"]].to_string(index=False))

                      word      R  % bị tách  % là entity
kiểu                                                     
VIẾT HOA TOÀN BỘ    1965.0  3.349      0.869        0.621
Viết hoa chữ đầu    8990.0  1.561      0.325        0.753
chữ thường         23727.0  1.104      0.067        0.004
có chữ số           5883.0  1.819      0.339        0.002
dấu câu / ký hiệu   5870.0  1.014      0.012        0.002 

              word      R  % bị tách  % là entity
độ dài từ                                        
1–4        27085.0  1.125      0.090        0.064
5–8        15226.0  1.559      0.269        0.339
9–12        3832.0  2.167      0.405        0.294
13+          292.0  3.260      0.634        0.291 

Word bị tách nhiều mảnh nhất:
                                 word    tag  pieces                                                                    tokens
http://www.pt.lu/infoweb/kreschtmaart      O      21 http : / / www . p ##t . l ##u / info ##we ##b / k ##res ##cht ##ma ##art
      

In [12]:
print("Ví dụ word thuộc entity bị tách (theo loại):")
for t in ["PER", "ORG", "LOC", "MISC"]:
    ex = W[(W.type == t) & (W.pieces > 1)].drop_duplicates("word").head(5)
    print(f"  {t}: " + " | ".join(f"{r.word} → {r.tokens}" for r in ex.itertuples()))
print("Ví dụ word O bị tách:", " | ".join(f"{r.word} → {r.tokens}" for r in W[(W.type == "O") & (W.pieces > 1) & (W.kiểu == 'chữ thường')].drop_duplicates("word").head(6).itertuples()))

Ví dụ word thuộc entity bị tách (theo loại):
  PER: CHINA → CH ##IN ##A | Nadim → Na ##di ##m | Ladki → La ##d ##ki | Shkvyrin → S ##h ##k ##vy ##rin | Oleg → Ole ##g
  ORG: RUGBY → R ##U ##GB ##Y | UNION → UN ##ION | Udinese → U ##dine ##se | Feyenoord → Fe ##ye ##no ##ord | LEEDS → L ##EE ##DS
  LOC: JAPAN → J ##AP ##AN | AL-AIN → AL - AI ##N | ITALY → IT ##AL ##Y | ROME → ROM ##E | Murrayfield → Murray ##field
  MISC: Uzbek → U ##z ##bek | Syrians → Syrian ##s | SKIING-WORLD → SK ##II ##NG - W ##OR ##LD | CUP → C ##UP | ASIAN → AS ##IA ##N
Ví dụ word O bị tách: newcomers → newcomer ##s | misdirected → mi ##s ##dir ##ec ##ted | lob → lo ##b | unstoppable → un ##stop ##pable | outsiders → outsider ##s | blunder → b ##lund ##er
